# BIO Filter Testing

Compare the saved FiNER-ORD vanilla teacher and CRF teacher predictions before and after BIO post-processing.

This notebook uses the same repair rule as `scripts/bio_repair.py` and reports:

- seed-level `entity_overall_f1` before and after repair
- aggregate mean and std for vanilla vs CRF
- whether BIO repair actually changes any of the current saved runs

In [2]:
import json
import sys
from pathlib import Path
from statistics import mean, stdev

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "evaluate.py").exists() and (candidate / "scripts" / "bio_repair.py").exists():
            return candidate
    raise RuntimeError(
        "Could not find repo root containing src/evaluate.py and scripts/bio_repair.py"
    )


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.evaluate import compute_detailed_metrics
from scripts.bio_repair import repair_bio

RESULTS_DIR = REPO_ROOT / "results"

RUN_FAMILIES = {
    "vanilla": [
        "phase_b_teacher_seed88",
        "phase_b_teacher_seed5768",
        "phase_b_teacher_seed78516",
    ],
    "crf": [
        "teacher_crf_seed88",
        "teacher_crf_seed5768",
        "teacher_crf_seed78516",
    ],
}

RESULTS_DIR


PosixPath('/Users/dayanbattulga/Desktop/personal-code/misc/dunedain_ner/results')

In [3]:
def load_predictions(run_id: str) -> dict:
    path = RESULTS_DIR / run_id / 'predictions.json'
    if not path.exists():
        raise FileNotFoundError(f'Missing predictions file: {path}')
    return json.loads(path.read_text())


def compute_before_after_metrics(run_id: str) -> dict:
    payload = load_predictions(run_id)
    true_labels = payload['true_labels']
    predictions = payload['predictions']
    repaired_predictions = [repair_bio(seq) for seq in predictions]

    before = compute_detailed_metrics(true_labels, predictions, verbose=False)
    after = compute_detailed_metrics(true_labels, repaired_predictions, verbose=False)

    return {
        'entity_overall_f1_before': before['entity_overall_f1'],
        'entity_overall_f1_after': after['entity_overall_f1'],
        'entity_overall_f1_delta': after['entity_overall_f1'] - before['entity_overall_f1'],
        'token_weighted_f1_before': before['token_weighted_f1'],
        'token_weighted_f1_after': after['token_weighted_f1'],
        'token_weighted_f1_delta': after['token_weighted_f1'] - before['token_weighted_f1'],
        'per_f1_before': before['entity_per_class'].get('PER', 0.0),
        'per_f1_after': after['entity_per_class'].get('PER', 0.0),
        'loc_f1_before': before['entity_per_class'].get('LOC', 0.0),
        'loc_f1_after': after['entity_per_class'].get('LOC', 0.0),
        'org_f1_before': before['entity_per_class'].get('ORG', 0.0),
        'org_f1_after': after['entity_per_class'].get('ORG', 0.0),
    }


def summarize_family(df: pd.DataFrame, family: str) -> dict:
    subset = df[df['family'] == family]
    before = subset['entity_overall_f1_before'].tolist()
    after = subset['entity_overall_f1_after'].tolist()
    return {
        'family': family,
        'n_runs': len(subset),
        'entity_overall_f1_mean_before': mean(before),
        'entity_overall_f1_std_before': stdev(before) if len(before) > 1 else 0.0,
        'entity_overall_f1_mean_after': mean(after),
        'entity_overall_f1_std_after': stdev(after) if len(after) > 1 else 0.0,
        'entity_overall_f1_mean_delta': mean(after) - mean(before),
    }


In [5]:
seed_rows = []
for family, run_ids in RUN_FAMILIES.items():
    for run_id in run_ids:
        metrics = compute_before_after_metrics(run_id)
        seed_rows.append(
            {
                'family': family,
                'run_id': run_id,
                **metrics,
            }
        )

seed_df = pd.DataFrame(seed_rows).sort_values(['family', 'run_id']).reset_index(drop=True)
display(seed_df)

seed_df[['family', 'run_id', 'entity_overall_f1_before', 'entity_overall_f1_after', 'entity_overall_f1_delta']]

,family,run_id,entity_overall_f1_before,entity_overall_f1_after,entity_overall_f1_delta,token_weighted_f1_before,token_weighted_f1_after,token_weighted_f1_delta,per_f1_before,per_f1_after,loc_f1_before,loc_f1_after,org_f1_before,org_f1_after
0,crf,teacher_crf_seed5768,0.850487,0.850487,0.0,0.985517,0.985135,-0.000382,0.948097,0.948097,0.854400,0.854400,0.799655,0.799655
1,crf,teacher_crf_seed78516,0.851883,0.851883,0.0,0.985186,0.985084,-0.000102,0.949565,0.949565,0.845426,0.845426,0.807790,0.807790
2,crf,teacher_crf_seed88,0.854027,0.854027,0.0,0.985306,0.984991,-0.000315,0.961672,0.961672,0.853081,0.853081,0.802039,0.802039
3,vanilla,phase_b_teacher_seed5768,0.850991,0.850991,0.0,0.985226,0.984952,-0.000274,0.956217,0.956217,0.844371,0.844371,0.802092,0.802092
4,vanilla,phase_b_teacher_seed78516,0.847530,0.847530,0.0,0.985004,0.984679,-0.000325,0.956063,0.956063,0.854400,0.854400,0.790295,0.790295
5,vanilla,phase_b_teacher_seed88,0.846832,0.846832,0.0,0.985251,0.984959,-0.000293,0.943060,0.943060,0.862682,0.862682,0.793677,0.793677


,family,run_id,entity_overall_f1_before,entity_overall_f1_after,entity_overall_f1_delta
0,crf,teacher_crf_seed5768,0.850487,0.850487,0.0
1,crf,teacher_crf_seed78516,0.851883,0.851883,0.0
2,crf,teacher_crf_seed88,0.854027,0.854027,0.0
3,vanilla,phase_b_teacher_seed5768,0.850991,0.850991,0.0
4,vanilla,phase_b_teacher_seed78516,0.847530,0.847530,0.0
5,vanilla,phase_b_teacher_seed88,0.846832,0.846832,0.0


In [6]:
family_df = pd.DataFrame(
    [summarize_family(seed_df, family) for family in RUN_FAMILIES]
).sort_values('family').reset_index(drop=True)

display(family_df)

family_df[['family', 'entity_overall_f1_mean_before', 'entity_overall_f1_mean_after', 'entity_overall_f1_mean_delta']]

,family,n_runs,entity_overall_f1_mean_before,entity_overall_f1_std_before,entity_overall_f1_mean_after,entity_overall_f1_std_after,entity_overall_f1_mean_delta
0,crf,3,0.852132,0.001783,0.852132,0.001783,0.0
1,vanilla,3,0.848451,0.002227,0.848451,0.002227,0.0


,family,entity_overall_f1_mean_before,entity_overall_f1_mean_after,entity_overall_f1_mean_delta
0,crf,0.852132,0.852132,0.0
1,vanilla,0.848451,0.848451,0.0
